In [4]:
import osmnx as ox
import networkx as nx

print("Localisation et téléchargement de la Scène...")

# On donne les adresses à Python
adresse_depart = "Hanoi University of Mining and Geology, Hanoi, Vietnam"
adresse_arrivee = "Hoa Binh Park, Hanoi, Vietnam"

# On récupère les coordonnées GPS exactes
point_A = ox.geocode(adresse_depart)
point_B = ox.geocode(adresse_arrivee)

# On télécharge le réseau routier 3km autour HUMG
graphe = ox.graph_from_point(point_A, dist=3000, network_type='drive')
print(f"Réseau téléchargé ! Nœuds (intersections) trouvés : {len(graphe.nodes)}")


print("Placement de l'Acteur sur la route...")

# On cherche donc l'intersection la plus proche
noeud_origine = ox.distance.nearest_nodes(graphe, X=point_A[1], Y=point_A[0])
noeud_destination = ox.distance.nearest_nodes(graphe, X=point_B[1], Y=point_B[0])


print("Calcul du chemin le plus court...")

# shortest Path
chemin_optimal = nx.shortest_path(graphe, source=noeud_origine, target=noeud_destination, weight='length')

# calcule la distance physique (en mètres) pour notre futur calcul de CO2 !
distance_m = nx.shortest_path_length(graphe, source=noeud_origine, target=noeud_destination, weight='length')
print(f"Distance totale du trajet : {distance_m / 1000:.2f} km")

route_gdf = ox.routing.route_to_gdf(graphe, chemin_optimal)

carte_finale = route_gdf.explore(
    color="red",
    style_kwds={"weight": 6, "opacity": 0.8},
    tiles="CartoDB positron",
    tooltip="name" # afficher le nom de la rue 
)

display(carte_finale)

Localisation et téléchargement de la Scène...
Réseau téléchargé ! Nœuds (intersections) trouvés : 2474
Placement de l'Acteur sur la route...
Calcul du chemin le plus court...
Distance totale du trajet : 3.06 km


In [5]:
import osmnx as ox
import geopandas as gpd
import folium

print("Téléchargement du réseau routier OSMnx (cela peut prendre quelques instants)...")

# 1. Définition des coordonnées et du périmètre
humg_coords = (21.0697, 105.7708)
rayon_m = 1500

# Téléchargement du graphe routier carrossable
G = ox.graph_from_point(humg_coords, dist=rayon_m, network_type='drive')

# Conversion du graphe en GeoDataFrame pour faciliter l'analyse spatiale
nodes, edges = ox.graph_to_gdfs(G)

print("Traitement et classification des types de routes...")

# 2. Définition des couleurs selon le type de route
# Note : L'attribut 'highway' d'OSMnx peut parfois être une liste de chaînes
def get_road_color(highway):
    if isinstance(highway, list):
        highway = highway[0]
        
    if highway == 'primary':
        return '#e74c3c'      # Rouge
    elif highway == 'tertiary':
        return '#f39c12'      # Orange
    elif highway == 'residential':
        return '#2ecc71'      # Vert
    elif highway == 'service':
        return '#3498db'      # Bleu
    else:
        return '#95a5a6'      # Gris (autres routes)

# Application des couleurs et de l'épaisseur de ligne
edges['color'] = edges['highway'].apply(get_road_color)
edges['weight'] = edges['highway'].apply(
    lambda x: 3.5 if (x[0] if isinstance(x, list) else x) == 'primary' else 2
)

# Optionnel : Filtrer strictement pour ne garder QUE les 4 types de routes
types_cibles = ['primary', 'tertiary', 'residential', 'service']
edges_filtrees = edges[edges['highway'].apply(lambda x: x[0] if isinstance(x, list) else x).isin(types_cibles)]

print("Génération de la carte professionnelle Folium...")

# 3. Initialisation de la carte avec un fond clair (CartoDB positron) pour un rendu "papier de recherche"
carte_zone = folium.Map(location=humg_coords, zoom_start=14, tiles='CartoDB positron')

# 4. Ajout du réseau routier colorisé
def style_function(feature):
    return {
        'color': feature['properties'].get('color', '#95a5a6'),
        'weight': feature['properties'].get('weight', 2),
        'opacity': 0.85
    }

folium.GeoJson(
    edges_filtrees[['geometry', 'color', 'weight', 'highway']].to_json(),
    name="Réseau Routier (4 types)",
    style_function=style_function,
    tooltip=folium.GeoJsonTooltip(fields=['highway'], aliases=['Type de route :'])
).add_to(carte_zone)

# 5. Ajout du périmètre d'étude (Cercle de 1,5 km)
folium.Circle(
    location=humg_coords,
    radius=rayon_m,
    color="#2c3e50",
    weight=2,
    dash_array='5, 5', # Bordure en pointillés
    fill=True,
    fill_opacity=0.05,
    name="Zone d'étude (Rayon 1.5 km)"
).add_to(carte_zone)

# 6. Ajout du marqueur central (Campus HUMG)
folium.Marker(
    location=humg_coords,
    popup=folium.Popup("<b>HUMG Campus</b><br>Centre de la zone d'étude", max_width=250),
    icon=folium.Icon(color='darkblue', icon='university', prefix='fa'),
    name="Campus HUMG"
).add_to(carte_zone)

# Ajout du contrôleur de couches
folium.LayerControl().add_to(carte_zone)

# Affichage du rendu final
display(carte_zone)
print("Carte générée avec succès !")

Téléchargement du réseau routier OSMnx (cela peut prendre quelques instants)...
Traitement et classification des types de routes...
Génération de la carte professionnelle Folium...


Carte générée avec succès !
